In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import Ridge
import gc
import random

# ==========================================
# 0. CONFIGURATION & DATA LOADING
# ==========================================
train_path = "../Datasets/ts-forecasting/train.parquet"
test_path = "../Datasets/ts-forecasting/train.parquet"

print("Loading data...")
train_raw = pd.read_parquet(train_path)
test_full = pd.read_parquet(test_path)

TARGET = "y_target"
FORECAST_WINDOWS = [1, 3, 10, 25]
N_FOLDS = 10
N_ITERATIONS = 20  # Set your random search loops here

# Hold out 20% of the training set globally upfront
print("Creating 80/20 Train/Holdout split...")
train_full, holdout_full = train_test_split(
    train_raw, test_size=0.2, random_state=42, shuffle=True
)

del train_raw; gc.collect()

def get_random_params():
    """True random sampling for the LightGBM config."""
    return {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': round(random.uniform(0.01, 0.1), 4),
        'n_estimators': 1500,
        'num_leaves': random.randint(31, 255),
        'min_child_samples': random.randint(20, 200),
        'feature_fraction': round(random.uniform(0.5, 1.0), 3),
        'verbosity': -1,
        'n_jobs': -1,
        'random_state': 42
    }

# ==========================================
# 1. HELPERS & METRICS
# ==========================================
def weighted_rmse_score(y_true, y_pred, w):
    """Custom metric for the competition."""
    if w is None:
        w = np.ones_like(y_true)
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true ** 2)) + 1e-8
    return float(np.sqrt(num / den))

# ==========================================
# 2. RANDOM SEARCH + EXACT 10-FOLD PIPELINE
# ==========================================
best_global_score = float('inf')
best_cfg = None

for iteration in range(N_ITERATIONS):
    lgb_cfg = get_random_params()
    print(f"\n{'*'*80}")
    print(f"🔄 RUN {iteration+1}/{N_ITERATIONS} | Config: {lgb_cfg}")
    print(f"{'*'*80}")
    
    test_outputs = []
    overall_holdout_y = []
    overall_holdout_pred = []
    overall_holdout_weights = []

    for horizon in FORECAST_WINDOWS:
        print(f"\n{'-'*40}")
        print(f" FORECAST HORIZON: {horizon}")
        print(f"{'-'*40}")
        
        tr_df = train_full[train_full['horizon'] == horizon].reset_index(drop=True)
        ho_df = holdout_full[holdout_full['horizon'] == horizon].reset_index(drop=True)
        te_df = test_full[test_full['horizon'] == horizon].reset_index(drop=True)

        if len(tr_df) == 0 or len(te_df) == 0 or len(ho_df) == 0:
            continue

        # Identify features (Added 'id' to drop_cols)
        drop_cols = [TARGET, 'ts_index', 'horizon', 'id']
        common_cols = list(set(tr_df.columns) & set(ho_df.columns) & set(te_df.columns))
        features = sorted([c for c in common_cols if c not in drop_cols])

        X_train = tr_df[features].copy()
        y_train = tr_df[TARGET]
        weights = tr_df['weight'].values if 'weight' in tr_df.columns else None
        
        X_holdout = ho_df[features].copy()
        y_holdout = ho_df[TARGET]
        weights_holdout = ho_df['weight'].values if 'weight' in ho_df.columns else None

        X_test = te_df[features].copy()

        # Handle Categoricals natively for LightGBM (Bulletproof method)
        # Exclude numbers and booleans, leaving only object/string/etc to be cast to category
        cat_cols = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()
        
        for col in cat_cols:
            X_train[col] = X_train[col].astype('category')
            X_holdout[col] = X_holdout[col].astype('category')
            X_test[col] = X_test[col].astype('category')

        # Arrays to store Base Model OOF, Holdout, and Test predictions
        oof_m1 = np.zeros(len(X_train))
        holdout_m1 = np.zeros(len(X_holdout))
        test_m1 = np.zeros(len(X_test))
        
        # 10-Fold Cross Validation Setup
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        
        for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train)):
            # Split Data
            X_trn, y_trn = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
            X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
            
            w_trn = weights[trn_idx] if weights is not None else None
            w_val = weights[val_idx] if weights is not None else None

            # --- Train Base Model ---
            m1 = lgb.LGBMRegressor(**lgb_cfg)
            m1.fit(
                X_trn, y_trn,
                sample_weight=w_trn,
                eval_set=[(X_val, y_val)],
                eval_sample_weight=[w_val] if w_val is not None else None,
                callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
            )
            oof_m1[val_idx] = m1.predict(X_val)
            holdout_m1 += m1.predict(X_holdout) / N_FOLDS
            test_m1 += m1.predict(X_test) / N_FOLDS
            
            # Quick In-fold tracking
            score_val = weighted_rmse_score(y_val.values, oof_m1[val_idx], w_val)
            print(f"Fold {fold+1:02d} | Base Model OOF WRMSE: {score_val:.5f}")

        # --- META MODEL (Level 1) ---
        X_meta_train = oof_m1.reshape(-1, 1)
        X_meta_holdout = holdout_m1.reshape(-1, 1)
        X_meta_test = test_m1.reshape(-1, 1)
        
        meta_model = Ridge(alpha=1.0, random_state=42)
        meta_model.fit(X_meta_train, y_train, sample_weight=weights)
        
        # Final Meta-Model Predictions
        ho_preds_blend = meta_model.predict(X_meta_holdout)
        test_preds_blend = meta_model.predict(X_meta_test)

        # Horizon Summary on the 20% Holdout
        hz_w_ho = weights_holdout if weights_holdout is not None else np.ones_like(y_holdout)
        horizon_ho_score = weighted_rmse_score(y_holdout.values, ho_preds_blend, hz_w_ho)
        print(f"\n>>> Horizon {horizon} Final 20% Holdout WRMSE: {horizon_ho_score:.5f}")
        
        # Store for global tracking
        overall_holdout_y.extend(y_holdout.values)
        overall_holdout_pred.extend(ho_preds_blend)
        overall_holdout_weights.extend(hz_w_ho)

        # Save Test Predictions
        te_df['prediction'] = test_preds_blend
        test_outputs.append(te_df[['id', 'prediction']])

        # Cleanup memory before next horizon
        del tr_df, ho_df, te_df, X_train, X_holdout, X_test, y_train, y_holdout
        gc.collect()

    # ==========================================
    # 3. EVALUATE RUN & SAVE BEST SUBMISSION
    # ==========================================
    run_global_score = weighted_rmse_score(
        np.array(overall_holdout_y),
        np.array(overall_holdout_pred),
        np.array(overall_holdout_weights)
    )

    print(f"\n{'='*40}")
    print(f"🏁 RUN {iteration+1} GLOBAL HOLDOUT WRMSE: {run_global_score:.5f}")
    print(f"{'='*40}\n")
    
    # If this run is the best so far, overwrite the submission file
    if run_global_score < best_global_score:
        best_global_score = run_global_score
        best_cfg = lgb_cfg
        print("   -> ⭐️ NEW BEST CONFIGURATION FOUND! Updating submission.csv...")
        
        submission = pd.concat(test_outputs, ignore_index=True)
        submission = submission[['id', 'prediction']]
        submission.dropna(subset=['prediction'], inplace=True)
        submission = submission.sort_values('id').reset_index(drop=True)
        submission.to_csv("submission.csv", index=False)
        print("   ✅ submission.csv successfully updated with new best predictions.")

print("\n" + "="*80)
print("🏆 ALL RANDOM SEARCH ITERATIONS COMPLETE")
print(f"Best Global 20% Holdout WRMSE: {best_global_score:.5f}")
print("Best Config:")
for k, v in best_cfg.items():
    print(f"    '{k}': {v},")
print("="*80)

Loading data...
Creating 80/20 Train/Holdout split...

********************************************************************************
🔄 RUN 1/20 | Config: {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.044, 'n_estimators': 1500, 'num_leaves': 194, 'min_child_samples': 31, 'feature_fraction': 0.78, 'verbosity': -1, 'n_jobs': -1, 'random_state': 42}
********************************************************************************

----------------------------------------
 FORECAST HORIZON: 1
----------------------------------------
